## Psychiatric Care Hospital Beds Over Time — EU & European Countries

**Data source:** Eurostat `HLTH_RS_BDS` — *Hospital beds by type of care (historical, 1960–2020)*  
**Variable:** `HBEDI_PSY` — Psychiatric care beds in hospitals (HP.1), absolute number  

**Design notes:**
- Only years where **≥ 5 countries** report data are shown (starts from 1975).
- Aggregate EU27/EU28 rows are excluded — country-level only.
- **Scandinavian countries** (Denmark 🇩🇰, Finland 🇫🇮, Norway 🇳🇴, Sweden 🇸🇪) are highlighted in amber; all others in steel blue.
- Use the **year slider** to jump to any year, or click **▶ Animate** to step through all years automatically.

In [1]:
# ── Dependencies ────────────────────────────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, clear_output
import time

# ── Load & clean data ────────────────────────────────────────────────────────
df_raw = pd.read_csv('./data/hlth_rs_bds_page_linear_2_0.csv')

df = (
    df_raw[['geo', 'Geopolitical entity (reporting)', 'TIME_PERIOD', 'OBS_VALUE']]
    .copy()
    .rename(columns={
        'geo': 'code',
        'Geopolitical entity (reporting)': 'country',
        'TIME_PERIOD': 'year',
        'OBS_VALUE': 'beds'
    })
    .dropna(subset=['beds'])
)

# Drop EU aggregates — keep individual countries only
EXCLUDE_CODES = {'EU27_2020', 'EU28'}
df = df[~df['code'].isin(EXCLUDE_CODES)].copy()
df['beds'] = pd.to_numeric(df['beds'], errors='coerce')
df = df.dropna(subset=['beds'])
df['year'] = df['year'].astype(int)

# ── Determine valid years (≥ 5 countries reporting) ─────────────────────────
country_count_per_year = df.groupby('year')['code'].nunique()
valid_years = sorted(country_count_per_year[country_count_per_year >= 5].index.tolist())

print(f"Data spans: {df['year'].min()} – {df['year'].max()}")
print(f"Valid years (≥5 countries): {valid_years[0]} – {valid_years[-1]}  "
      f"({len(valid_years)} years)")
print(f"Total countries in dataset: {df['code'].nunique()}")

Data spans: 1965 – 2020
Valid years (≥5 countries): 1975 – 2019  (45 years)
Total countries in dataset: 37


In [2]:
# ── Styling constants ────────────────────────────────────────────────────────
SCANDI_CODES   = {'DK', 'FI', 'NO', 'SE'}   # Scandinavian countries
COLOR_SCANDI   = '#E8A838'                   # Warm amber
COLOR_OTHER    = '#4A7FB5'                   # Steel blue
COLOR_BG       = '#F7F9FC'                   # Light background
COLOR_GRID     = '#DDEAF5'

def plot_year(year: int, ax=None, fig=None):
    """Draw the horizontal bar chart for a single year."""
    year_df = (
        df[df['year'] == year]
        .sort_values('beds', ascending=True)
        .reset_index(drop=True)
    )

    n = len(year_df)
    bar_height = max(0.55, min(0.82, 14 / max(n, 1)))
    fig_height = max(5, n * 0.38 + 2)

    if ax is None:
        fig, ax = plt.subplots(figsize=(13, fig_height))

    ax.clear()
    fig.patch.set_facecolor(COLOR_BG)
    ax.set_facecolor(COLOR_BG)

    colors = [COLOR_SCANDI if c in SCANDI_CODES else COLOR_OTHER
              for c in year_df['code']]

    bars = ax.barh(
        year_df['country'], year_df['beds'],
        color=colors, height=bar_height,
        edgecolor='white', linewidth=0.4
    )

    # Value labels
    max_val = year_df['beds'].max()
    for bar, val in zip(bars, year_df['beds']):
        offset = max_val * 0.012
        ax.text(
            val + offset, bar.get_y() + bar.get_height() / 2,
            f'{int(val):,}',
            va='center', ha='left',
            fontsize=7.5, color='#333333', fontweight='500'
        )

    # Bold Scandinavian country labels
    for label, code in zip(ax.get_yticklabels(), year_df['code']):
        if code in SCANDI_CODES:
            label.set_fontweight('bold')
            label.set_color('#B07010')

    ax.set_xlabel('Number of Psychiatric Care Beds', fontsize=11, labelpad=8)
    ax.set_title(
        f'Psychiatric Care Hospital Beds — {year}\n'
        f'({n} countries reporting)',
        fontsize=13, fontweight='bold', pad=12, color='#1A2A3A'
    )

    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
    ax.tick_params(axis='y', labelsize=8.5)
    ax.tick_params(axis='x', labelsize=9)
    ax.set_xlim(0, max_val * 1.15)

    ax.grid(axis='x', color=COLOR_GRID, linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_visible(False)

    # Legend
    legend_elements = [
        mpatches.Patch(color=COLOR_SCANDI, label='Scandinavian (DK, FI, NO, SE)'),
        mpatches.Patch(color=COLOR_OTHER,  label='Other European countries'),
    ]
    ax.legend(
        handles=legend_elements,
        loc='lower right', fontsize=9,
        framealpha=0.85, edgecolor='#CCCCCC'
    )

    # Source annotation
    fig.text(
        0.99, 0.01,
        'Source: Eurostat HLTH_RS_BDS | Psychiatric care beds (HBEDI_PSY)',
        ha='right', va='bottom', fontsize=7, color='#888888'
    )

    fig.tight_layout(rect=[0, 0.02, 1, 1])
    return fig, ax

In [3]:
# ── Interactive widget ────────────────────────────────────────────────────────
#
#  Controls:
#    • Year slider  – jump to any valid year
#    • ▶ Animate    – step automatically through all valid years
#    • ■ Stop        – pause the animation mid-run
#    • Speed slider – animation interval in seconds (0.3 – 3.0 s)
# ─────────────────────────────────────────────────────────────────────────────

year_slider = widgets.SelectionSlider(
    options=valid_years,
    value=valid_years[0],
    description='Year:',
    style={'description_width': '40px'},
    layout=widgets.Layout(width='680px')
)

speed_slider = widgets.FloatSlider(
    value=0.7,
    min=0.3, max=3.0, step=0.1,
    description='Speed (s):',
    style={'description_width': '70px'},
    layout=widgets.Layout(width='340px'),
    readout_format='.1f'
)

btn_play = widgets.Button(
    description='▶  Animate',
    button_style='success',
    layout=widgets.Layout(width='120px', height='34px')
)
btn_stop = widgets.Button(
    description='■  Stop',
    button_style='danger',
    layout=widgets.Layout(width='100px', height='34px'),
    disabled=True
)

out = widgets.Output()

# State flag
_state = {'running': False}

# Pre-create a figure that gets reused (prevents repeated figure creation)
_fig, _ax = plt.subplots(figsize=(13, 8))
plt.close(_fig)  # Don't display yet


def render(year):
    """Render chart for given year into the output widget."""
    with out:
        clear_output(wait=True)
        fig, ax = plot_year(year)
        plt.show()


def on_slider_change(change):
    if not _state['running']:
        render(change['new'])


def on_play(_):
    _state['running'] = True
    btn_play.disabled = True
    btn_stop.disabled = False
    year_slider.disabled = True

    for year in valid_years:
        if not _state['running']:
            break
        year_slider.value = year
        render(year)
        time.sleep(speed_slider.value)

    _state['running'] = False
    btn_play.disabled = False
    btn_stop.disabled = True
    year_slider.disabled = False


def on_stop(_):
    _state['running'] = False
    btn_stop.disabled = True
    btn_play.disabled = False
    year_slider.disabled = False


year_slider.observe(on_slider_change, names='value')
btn_play.on_click(on_play)
btn_stop.on_click(on_stop)

controls = widgets.HBox([
    btn_play, btn_stop,
    widgets.VBox([year_slider, speed_slider])
], layout=widgets.Layout(align_items='center', gap='16px'))

display(controls, out)

# Render the initial frame
render(valid_years[0])

Output()